*Note: The code in this notebook was generated with the assistance of Claude.*

## Multinomial Logistic Regression and KNN

We are going to do some analysis for the wine dataset. https://archive.ics.uci.edu/dataset/109/wine

These data are the results of a chemical analysis of wines grown in the same region of Italy, but derived from different cultivars. (A cultiaver is a cultivated plant that people have bred/selected for different traits. It is a portmanteau of "cultivated variety". See wikipedia for more details: https://en.wikipedia.org/wiki/Cultivar)

Our goal is to predict the **cultivar** (3 classes) of 178 wines from 13 chemical measurements. We will do this first using multinomial logistic regression and then KNN. 

Game plan:
1. Hold out a 20% test set.
2. On the training set, use stratified 5-fold CV to choose `C` for **lasso** multinomial logistic regression.
3. Use the same folds to choose `k` for **KNN**.
4. Compare the two models, then evaluate each **once** on the test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import log_loss, ConfusionMatrixDisplay

### Load the data

Note: Normally, we upload the data from a .csv file in our data sub directory. In this instance, we are using the wine dataset, which is included as a dataset in sklearn. Some of the other datasets we have used in the past are also available in sklearn, but I've chosen to include them as .csv files to practice ingesting data. 

In [ ]:
# Let's load and explore the data. 
# Note that the data takes a slightly different form than a standard .csv, but is well documented in sklearn

wine = load_wine(as_frame=True)
X = wine.data
y = wine.target   # 0, 1, 2 = the three cultivars

print(X.shape)
y.value_counts().sort_index()

In [ ]:
X.head()

In [ ]:
# Let's look at the ranges, mean, and standard deviation of each of these numeric variables
X.describe().T[["mean", "std", "min", "max"]].round(2)

All 13 predictors are numeric, but on very different scales (e.g. `proline` is in the hundreds, `hue` is around 1). Both lasso and KNN depend on scale, so we standardize everything.

### Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=113
)
print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

### CV folds and a baseline

In [ ]:
# Same folds for every model so comparisons are fair
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=224)

# Baseline: predict the training class proportions for every wine
class_props = y_train.value_counts(normalize=True).sort_index().values

#np.tile (class_props, (len(y_train), 1)) gives the baseline probability for every class
ll_base = log_loss(y_train, np.tile(class_props, (len(y_train), 1)))
print(f"Base-rate log loss: {ll_base:.4f}")

In [ ]:
# Calculating log_loss by hand
y_ll = pd.get_dummies(y_train).sort_index(axis=1).values 
p_ll = np.tile(class_props, (len(y_train), 1)) 
ll_by_hand = -np.mean(np.sum(y_ll * np.log(p_ll), axis=1))
ll_by_hand

In [ ]:
# We can look at each of the elements above individually to see what is going on. 
# Equation for log loss for multinomial regression is on slide 56 of Classification_and_GLMs.pptx

#np.tile(class_props, (len(y_train), 1))
#p_ll
#y_ll
#y_ll * np.log(p_ll)
#np.sum(y_ll * np.log(p_ll), axis=1)


**Helper functions**
* Similar to what we used last week 
* `cv_scores`: log_loss: mean and SD across the 5 folds.
* `cv_pred`: out-of-fold predicted class for every training wine (for a pooled confusion matrix).

In [ ]:
def cv_scores(model):
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=["neg_log_loss"])
    return {
        "log loss (mean)": -res["test_neg_log_loss"].mean(),
        "log loss (sd)": res["test_neg_log_loss"].std(),
    }

def cv_pred(model):
    return cross_val_predict(model, X_train, y_train, cv=cv)

## Lasso multinomial logistic regression

With 3 classes, `LogisticRegression` fits a **multinomial** model: one set of 13 coefficients per class (39 total coefficeints), and a softmax turns the three linear scores into probabilities that sum to 1.

Lasso: `l1_ratio=1` with `solver="saga"`. Remember: small `C` = strong penalty.

In [ ]:
def make_model(C, l1_ratio):
    return Pipeline(steps=[
        ("scale", StandardScaler()),
        ("model", LogisticRegression(C=C, l1_ratio=l1_ratio, solver="saga", max_iter=50000, random_state=48)),
    ])

C_values = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]
l1_ratios = {"lasso": 1, "enet 0.75": 0.75, "enet 0.5": 0.5, "enet 0.25": 0.25, "ridge": 0}

all_results = pd.DataFrame([{"model": name, "C": C, **cv_scores(make_model(C, r))}
                            for name, r in l1_ratios.items() for C in C_values])
all_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, g in all_results.groupby("model"):
    ax.errorbar(g["C"], g["log loss (mean)"], marker="o", capsize=3, label=name)
ax.axhline(ll_base, color="red", ls=":", label="base rate (no predictors)")
ax.set(xscale="log", xlabel="C (log scale; smaller = stronger penalty)",
       ylabel="mean CV log loss (± 1 SD across folds)", title="CV log loss vs. C")
ax.legend()
plt.show()

In [ ]:
# From all of the values of C that we tried above, let's select the best one
# We will identify the simplest model that is within one standard error of the lowest
best = all_results.loc[all_results["log loss (mean)"].idxmin()]
cutoff = best["log loss (mean)"] + best["log loss (sd)"] / np.sqrt(5)   

candidates = all_results[all_results["log loss (mean)"] <= cutoff].copy()
candidates["l1_ratio"] = candidates["model"].map(l1_ratios)

# Fit each candidate on the full training set and count its nonzero coefficients
candidates["nonzero coefs (of 39)"] = [
    int((make_model(C, r).fit(X_train, y_train).named_steps["model"].coef_ != 0).sum())
    for C, r in zip(candidates["C"], candidates["l1_ratio"])
]
simplest = candidates.sort_values(["nonzero coefs (of 39)", "C"]).iloc[0]   # fewest nonzero, then smallest C

C_best, method_best, l1_best = simplest["C"], simplest["model"], simplest["l1_ratio"]
print(f"Simplest within 1 SE: {method_best} (l1_ratio = {l1_best}), C = {C_best}")
model_best = make_model(C_best, l1_best)
candidates.round(4)

### Coefficients of the chosen lasso model

In [ ]:
# Now that we have a sense of the C value that produces the best model
# let's look more specifically at the coefficients it picks out. 

model_best = make_model(C_best, l1_best)
model_best.fit(X_train, y_train)
coef_table = pd.DataFrame(model_best.named_steps["model"].coef_.T,
                          index=X.columns, columns=wine.target_names)

sns.heatmap(coef_table, annot=True, fmt=".2f", cmap="RdBu", center=0)
plt.title(f"Lasso coefficients (C = {C_best}), standardized predictors")
plt.show()

### Commnents on the coefficients

Remember that there is no reference level for multi-nomial classification, all levels are treated equally. So coefficients help us compare two of the three levels. 

Example: Proline. 
* An increase of 1 SD multiplies the odds of class 0 vs class 1 by $\mathrm{e}^{(1.09-(-2.45))}= \mathrm{e}^{3.54} \approx 34$ and class 0 vs class 2 by $\mathrm{e}^{1.09} = \approx 3$

### Confusion matrix (pooled out-of-fold predictions)

In [ ]:
pred_lasso = cv_pred(model_best)
ConfusionMatrixDisplay.from_predictions(y_train, pred_lasso, cmap="Blues", colorbar=False)
plt.title(f"Lasso (C = {C_best}), 5-fold CV")
plt.show()

## K-Nearest Neighbors

KNN has no coefficients and no penalty. It predicts a wine's class by a vote of its `k` nearest training wines (Euclidean distance on the standardized predictors). The predicted probability for a class is the fraction of the `k` neighbors in that class.

We choose `k` with the same folds and the same criterion (lowest mean CV log loss).

In [ ]:
def make_knn(k):
    return Pipeline(steps=[
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k)),
    ])

k_values = range(1, 41, 2)   # odd k avoids some ties

# Note: We are using the same cv_scores function that we used for the multi-nomial logistic regression
knn_results = pd.DataFrame([{"k": k, **cv_scores(make_knn(k))} for k in k_values])
knn_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(knn_results["k"], knn_results["log loss (mean)"], yerr=knn_results["log loss (sd)"],
            marker="o", capsize=3)
ax.axhline(ll_base, color="red", ls=":", label="base rate (no predictors)")
ax.set(xlabel="k", ylabel="mean CV log loss (± 1 SD)", title="KNN: CV log loss vs. k")
ax.legend()
plt.show()

In [ ]:
k_best = int(knn_results.loc[knn_results["log loss (mean)"].idxmin(), "k"])
print(f"Best k (lowest mean CV log loss): {k_best}")
model_knn = make_knn(k_best)

Comments:
* **Small `k`:** log loss is huge. With `k = 1` every predicted probability is 0 or 1 (i.e. each model assigns probability 1 to the nearest neighbor's class, and 0 to the other classes.) Each mistake gives the true class a probability of zero, which log loss penalizes heavily.
* **Large `k`:** log loss rises steadily because each prediction averages over more distant neighbors.


In [ ]:
pred_knn = cv_pred(model_knn)
ConfusionMatrixDisplay.from_predictions(y_train, pred_knn, cmap="Blues", colorbar=False)
plt.title(f"KNN (k = {k_best}), 5-fold CV")
plt.show()

## Comparing lasso and KNN

In [ ]:
summary = pd.DataFrame({
    f"Lasso (C = {C_best})": cv_scores(model_best),
    f"KNN (k = {k_best})": cv_scores(model_knn),
}).T
summary.round(4)

In [ ]:
# Which training wines does each model get wrong (out-of-fold)?
compare = pd.DataFrame({"actual": y_train, "lasso": pred_lasso, "knn": pred_knn})
wrong = compare[(compare["lasso"] != compare["actual"]) | (compare["knn"] != compare["actual"])]
print(f"Models agree on {(pred_lasso == pred_knn).mean():.1%} of training wines")
wrong

How the answers are the same and different:
* **Same:** the two models agree on 98% of training wines. Every mistake either model makes is on a class 1 wine, and both models miss the same two wines (rows 73 and 83).
* **Different (performance):** lasso has a lower CV log loss (0.030 vs. 0.117) and misclassifies 2 wines vs. 5 for KNN.
* **Different (interpretation):** lasso tells us which features matter for which class, and drops 2 features. KNN has no coefficients and uses all 13 (standardized) features equally in its distances, including ones lasso found unnecessary.
* **Different (shape of the boundary):** lasso's class boundaries are linear in the predictors; KNN's can be any shape. Here, linear boundaries are enough.

## Refit on all training data and test once on the holdout set

In [ ]:
model_best.fit(X_train, y_train)
model_knn.fit(X_train, y_train)

test_results = pd.DataFrame({
    name: {
        "test log loss": log_loss(y_test, m.predict_proba(X_test)),
    }
    for name, m in [(f"Lasso (C = {C_best})", model_best), (f"KNN (k = {k_best})", model_knn)]
}).T
test_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, m) in zip(axes, [(f"Lasso (C = {C_best})", model_best), (f"KNN (k = {k_best})", model_knn)]):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_test), ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}, test set")
plt.tight_layout()
plt.show()

Comments on the holdout results:
* Both models beat the base rate (log loss 1.085) by a wide margin on new data.
* Lasso still has the lower log loss. 